# Ordered Logistic Regression Results: FAIR^2 Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 rangeland management dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant schema provided at:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

We'll inspect its structure, extract record sets, and conduct exploratory data analysis step by step.

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`. We'll print a summary from its metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (do not treat as dict)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")
if hasattr(metadata, 'license'):
    print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
List all available record sets and fields, referencing their `@id`.

- **Record sets** correspond to logical tables or resources in the dataset.
- **Fields** represent columns within each record set.

We'll enumerate all record sets and their fields by their `@id`.

In [ ]:
# Find all record sets in the metadata.
record_sets = dataset.record_sets()
print("Available record sets (@id and name):\n")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")

print("\nRecord sets and their fields:\n")
for rs in record_sets:
    if 'fields' in rs:
        print(f"Record set: {rs['@id']}")
        for field in rs['fields']:
            print(f"  - Field @id: {field['@id']} | name: {field.get('name', '(no name)')}")
    else:
        print(f"Record set: {rs['@id']} (Fields not listed in metadata)")

### Example Record Preview
Below we print a few sample records from each record set, referencing them by their `@id`.

In [ ]:
# Print up to 2 records from each record set, referenced by @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample records from record set '@id': {rs_id}")
    try:
        for i, x in enumerate(dataset.records(record_set=rs_id)):
            if i >= 2:
                break
            print(x)
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load all available record sets into DataFrames for further analysis. Always reference by each record set's `@id`.

In [ ]:
# Gather @ids for all record sets we discovered above
all_record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load each record set to a DataFrame, referenced by @id
for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Show columns for first non-empty record set
for rs_id, df in dataframes.items():
    if not df.empty:
        print(f"\nColumns in record set '@id': {rs_id}")
        print(df.columns.tolist())
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field in a record set (referenced by its field `@id`) for filtering and normalization. If no numeric field is obvious, adapt as appropriate.

In [ ]:
# Example: use the first record set with a numeric field for EDA.
numeric_field = None
group_field = None
record_set_id = None

# Try to automatically find a record set with at least one numeric field
for rs_id, df in dataframes.items():
    if df.empty:
        continue
    # Try to find a numeric column
    for col in df.select_dtypes(include=['number']).columns:
        numeric_field = col
        record_set_id = rs_id
        # Select a group field if one exists
        for c in df.columns:
            if c != col and (df[c].nunique() > 1 and df[c].dtype == object):
                group_field = c
                break
        break
    if numeric_field and record_set_id:
        break

if not numeric_field:
    print("No numeric fields found for EDA.")
else:
    print(f"Performing EDA on record set '@id': {record_set_id}, numeric field: {numeric_field}")
    df = dataframes[record_set_id]
    # Example threshold–here, we use mean + 1 std as an example cutoff
    threshold = df[numeric_field].mean() + df[numeric_field].std()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}.")
    # Z-normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:\n", grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships. We'll use pandas and matplotlib for a simple plot.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of the numeric field, if found
if numeric_field and record_set_id:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using `mlcroissant`, referenced all entities by their `@id`, and performed simple extraction and EDA.

- We loaded the metadata and discovered available record sets and fields via their Croissant `@id`s.
- We extracted records, demonstrated basic filtering and normalization, and visualized field distributions.
- This approach forms a template for reproducible, schema-driven exploration of FAIR datasets.

Further steps can include advanced analysis or model-building on top of the normalized and grouped data.